In [ ]:
# ============================================================
# CELL 0: CẤU HÌNH PHIÊN (HƯỚNG 3: DỮ LIỆU GỘP CHIA 3 TẬP)
# ============================================================
MODEL = "yolo"          # "yolo" = YOLO26s-seg | "tsvm" = YOLO26s-seg + TSVM
SEEDS = [7]            # các seed chạy lần lượt trong phiên này (tổng thể: seed 0–9)
CHAY_THU = False       # True: 1 epoch, 1 seed, để kiểm tra notebook trước khi chạy thật
GIOI_HAN_GIO = 11.8        # mốc an toàn: Kaggle cắt phiên ở 12 giờ
DU_PHONG_GIO = 0.25        # để dành cho đánh giá test + nén zip sau khi dừng huấn luyện

assert MODEL in ("yolo", "tsvm")
assert all(s in range(10) for s in SEEDS), "Hướng 3 chỉ dùng seed 0–9"
EPOCHS = 1 if CHAY_THU else 100
if CHAY_THU:
    SEEDS = SEEDS[:1]
print(f"MODEL={MODEL}, SEEDS={SEEDS}, EPOCHS={EPOCHS}")


In [ ]:
# ============================================================
# CELL 1: TÌM INPUT, NẠP ULTRALYTICS BẢN TSVM_FP32, TẠO DATA YAML (TRAIN / VAL / TEST)
# ============================================================
import os, sys, glob, time, json, shutil, yaml
T_PHIEN = time.time()

def tim(mau):
    return sorted(glob.glob(mau, recursive=True))

_repo = tim("/kaggle/input/**/ultralytics_TSVM_FP32/PATCH_FP32.md")
_data = tim("/kaggle/input/**/H3_Gop_3tap/images/test")
assert _repo, "❌ Chưa thêm dataset chứa thư mục ultralytics_TSVM_FP32"
assert _data, "❌ Chưa thêm dataset h3-gop-3tap-tsvm (thư mục H3_Gop_3tap)"
CUSTOM_REPO_PATH = os.path.dirname(_repo[0])
DATASET_ROOT = os.path.dirname(os.path.dirname(_data[0]))

if not os.path.exists("/kaggle/working/ultralytics"):
    os.symlink(CUSTOM_REPO_PATH, "/kaggle/working/ultralytics")
for p in ["/kaggle/working", CUSTOM_REPO_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

import ultralytics
assert ultralytics.__version__ == "8.4.127", ultralytics.__version__
assert ultralytics.__file__.startswith("/kaggle/working/ultralytics"), ultralytics.__file__
from ultralytics.nn.modules import C2TSVMamba
print(f"✅ Ultralytics {ultralytics.__version__} (bản TSVM_FP32) từ {ultralytics.__file__}")

dem = {s: len(os.listdir(f"{DATASET_ROOT}/images/{s}")) for s in ("train", "val", "test")}
dem_nhan = {s: len(os.listdir(f"{DATASET_ROOT}/labels/{s}")) for s in ("train", "val", "test")}
assert dem == {"train": 2455, "val": 532, "test": 522} and dem_nhan == {"train": 2455, "val": 532, "test": 522}, (dem, dem_nhan)
print("📁 Dữ liệu:", DATASET_ROOT, dem)

DATA_YAML = "/kaggle/working/data_h3_3tap.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump({"path": DATASET_ROOT, "train": "images/train", "val": "images/val", "test": "images/test",
                    "nc": 1, "names": {0: "polyp"}}, f, sort_keys=False, allow_unicode=True)

if MODEL == "tsvm":
    SRC_YAML = f"{CUSTOM_REPO_PATH}/cfg/models/26/yolo26-seg-TopologyShapeVMamba.yaml"
    # Tên tệp PHẢI bắt đầu bằng "yolo26s": Ultralytics đoán scale từ tên tệp
    MODEL_YAML = "/kaggle/working/yolo26s-seg-TopologyShapeVMamba.yaml"
    with open(SRC_YAML, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    cfg["scale"] = "s"
    with open(MODEL_YAML, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    TEN = "TSVM_FP32"
else:
    MODEL_YAML = None
    TEN = "YOLO26s_seg"


In [ ]:
# ============================================================
# CELL 2: TẢI TRỌNG SỐ yolo26s-seg.pt (CẦN Settings > Internet: ON)
# ============================================================
WEIGHT_PATH = "/kaggle/working/yolo26s-seg.pt"
URL = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26s-seg.pt"
if os.path.exists(WEIGHT_PATH) and os.path.getsize(WEIGHT_PATH) < 20 * 1024 * 1024:
    os.remove(WEIGHT_PATH)
if not os.path.exists(WEIGHT_PATH):
    !curl -L --retry 5 --retry-delay 2 -C - "{URL}" -o "{WEIGHT_PATH}"
size_mb = os.path.getsize(WEIGHT_PATH) / 2**20
print(f"Dung lượng: {size_mb:.2f} MB")
assert size_mb > 20.0, "❌ Tải chưa đủ, chạy lại cell này (Settings > Internet: ON)"


In [ ]:
# ============================================================
# CELL 3: KIỂM TRA CẤU TRÚC MÔ HÌNH TRƯỚC KHI TRAIN (~10 GIÂY)
# ============================================================
from ultralytics import YOLO
_m = YOLO(MODEL_YAML).load(WEIGHT_PATH) if MODEL == "tsvm" else YOLO(WEIGHT_PATH)
l10 = _m.model.model[10]
n_all = sum(p.numel() for p in _m.model.parameters())
n_l10 = sum(p.numel() for p in l10.parameters())
print(f"Tầng 10: {type(l10).__name__}, {n_l10:,} tham số; toàn mô hình {n_all:,}")
if MODEL == "tsvm":
    assert type(l10).__name__ == "C2TSVMamba" and n_l10 == 1_811_201, "❌ Tầng 10 không đúng TSVM đầy đủ"
else:
    assert type(l10).__name__ == "C2PSA", "❌ Baseline phải giữ C2PSA ở tầng 10"
print("✅ Đúng cấu trúc")
del _m


In [ ]:
# ============================================================
# CELL 4: HUẤN LUYỆN TỪNG SEED (GIỐNG HỆT BÁO CÁO) + MỐC AN TOÀN 11,8 GIỜ + ĐÁNH GIÁ TRÊN TẬP TEST
# ============================================================
import random, numpy as np, torch
WORKERS, BATCH = 2, 8
thoi_gian_seed, bo_qua, dung_som = [], [], []

def gio_phien():
    return (time.time() - T_PHIEN) / 3600

def tao_callback_dung(run_dir_ref, trang_thai):
    """Cuối mỗi epoch (sau khi đã lưu last.pt): nếu epoch kế tiếp có thể làm phiên vượt mốc thì dừng."""
    def cb(trainer):
        if trang_thai["da_dung"]:
            return
        now = time.time()
        ep = now - trang_thai["t_epoch"]
        trang_thai["t_epoch"] = now
        if gio_phien() + 1.1 * ep / 3600 + DU_PHONG_GIO > GIOI_HAN_GIO and trainer.epoch + 1 < trainer.epochs:
            trang_thai["da_dung"] = True
            trainer.stop = True
            shutil.copy2(trainer.last, trainer.last.parent / "last_resume.pt")   # last.pt sắp bị bỏ optimizer
            with open(f"{trainer.save_dir}/CHUA_XONG.txt", "w") as f:
                f.write(f"Dừng ở mốc an toàn sau epoch {trainer.epoch + 1}/{trainer.epochs}, phiên {gio_phien():.2f} giờ.\n"
                        "Chạy tiếp bằng notebook h3-chay-tiep với weights/last_resume.pt.\n")
            print(f"⏸ DỪNG AN TOÀN sau epoch {trainer.epoch + 1}: phiên {gio_phien():.2f} giờ, epoch vừa rồi {ep / 60:.1f} phút")
    return cb

for SEED in SEEDS:
    du_kien = max(thoi_gian_seed) * 1.1 if thoi_gian_seed else 0.0
    if thoi_gian_seed and gio_phien() + du_kien + DU_PHONG_GIO > GIOI_HAN_GIO:
        print(f"⏸ Bỏ qua seed {SEED}: phiên đã {gio_phien():.1f} giờ, seed trước {du_kien / 1.1:.1f} giờ. Chạy seed này ở phiên khác.")
        bo_qua.append(SEED)
        continue
    t0 = time.time()
    os.environ["PYTHONHASHSEED"] = str(SEED)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    torch.cuda.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=False)

    model = YOLO(MODEL_YAML).load(WEIGHT_PATH) if MODEL == "tsvm" else YOLO(WEIGHT_PATH)
    RUN_NAME = f"H3_Gop_3tap_{TEN}_s{SEED}" + ("_chaythu" if CHAY_THU else "")
    trang_thai = {"da_dung": False, "t_epoch": time.time()}
    model.add_callback("on_train_start", lambda tr: trang_thai.update(t_epoch=time.time()))
    model.add_callback("on_fit_epoch_end", tao_callback_dung(None, trang_thai))
    model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=BATCH, device=0, workers=WORKERS, cache=False,
                amp=False, optimizer="AdamW", seed=SEED, lr0=0.001, warmup_epochs=5.0, deterministic=True,
                patience=100, close_mosaic=10, project="/kaggle/working/runs", name=RUN_NAME, exist_ok=True, plots=True)
    run_dir = f"/kaggle/working/runs/{RUN_NAME}"
    del model
    torch.cuda.empty_cache()
    if trang_thai["da_dung"]:
        dung_som.append(SEED)

    # Đánh giá best.pt trên tập TEST (chỉ một lần, sau khi đã chọn checkpoint trên val). Seed dừng sớm vẫn đánh giá
    # để có số tạm, nhưng số chính thức lấy sau khi chạy tiếp đủ 100 epoch.
    best = YOLO(f"{run_dir}/weights/best.pt")
    m = best.val(data=DATA_YAML, split="test", imgsz=640, batch=BATCH, half=False, device=0, plots=False,
                 project="/kaggle/working/runs", name=RUN_NAME + "_test", exist_ok=True)
    kq = {k: float(v) for k, v in m.results_dict.items()}
    kq.update(seed=SEED, model=TEN, gio_huan_luyen=(time.time() - t0) / 3600, dung_som=trang_thai["da_dung"],
              best_pt_mb=os.path.getsize(f"{run_dir}/weights/best.pt") / 2**20,
              gpu=torch.cuda.get_device_name(0), torch=torch.__version__)
    with open(f"{run_dir}/test_metrics.json", "w") as f:
        json.dump(kq, f, indent=1)
    print(f"✅ seed {SEED}: test mask mAP50-95 {kq.get('metrics/mAP50-95(M)', float('nan')):.4f}, "
          f"{kq['gio_huan_luyen']:.2f} giờ" + ("  (DỪNG SỚM, cần chạy tiếp)" if trang_thai["da_dung"] else ""))
    del best
    torch.cuda.empty_cache()

    shutil.make_archive(f"/kaggle/working/runs_{RUN_NAME}", "zip", "/kaggle/working/runs", RUN_NAME)
    thoi_gian_seed.append((time.time() - t0) / 3600)

print(f"\nXong phiên. Đã chạy: {[s for s in SEEDS if s not in bo_qua]}; bỏ qua: {bo_qua}; dừng sớm: {dung_som}; "
      f"tổng {gio_phien():.2f} giờ")
